# 01장. NEIS API 요청하기

| 학습 질문 |
|---|
| 우리 학교 급식을 공식 서버에 어떻게 요청할까? |


## 이 장에서 배울 내용

- API의 요청과 응답을 식당 주문에 비유해 설명할 수 있다.
- 요청 주소와 조건을 확인한 뒤 NEIS API에 GET 요청을 보낼 수 있다.
- 상태 코드와 받은 행 수로 요청 결과를 확인할 수 있다.
- 학교명보다 학교 코드가 정확한 식별값인 이유를 말할 수 있다.
- 연결에 실패했을 때 같은 기간의 예비 자료로 전환할 수 있다.


## 생각 열기

부록 A에서 JSON 파일을 여는 방법을 익혔습니다. 이제 자료를 이미 저장해 둔 파일에서 읽는 대신, NEIS 서버의 정해진 창구에 학교 코드와 날짜를 보내 직접 받아 봅시다.


## 핵심 용어

| 용어 | 뜻 |
|---|---|
| **API** | 다른 서비스에 정해진 규칙으로 데이터를 요청하는 창구 |
| **요청** | 주소와 조건을 서버에 보내는 일 |
| **응답** | 서버가 요청을 처리한 뒤 돌려주는 결과 |
| **params** | 어느 학교·기간·형식의 자료를 원하는지 적은 요청 조건 |
| **GET** | 서버에 자료를 달라고 요청하는 대표적인 HTTP 방식 |
| **상태 코드** | 서버가 요청을 어떻게 처리했는지 나타내는 숫자 |
| **학교 코드** | 같은 이름의 학교를 구분하는 공식 식별값 |


## 개념 익히기


API는 급식실 주문 창구와 비슷합니다. ‘남악고등학교, 2026년 6월 24일부터 30일까지 중식’을 정해진 양식으로 요청하면 서버가 정해진 양식의 응답을 줍니다.

### 1. 주소와 조건을 나누어 적는다

요청 주소는 `https://open.neis.go.kr/hub/mealServiceDietInfo`입니다. 교육청 코드, 학교 코드, 날짜 같은 조건은 `params` 사전에 넣습니다. `requests.get()`은 주소와 조건을 합쳐 GET 요청을 보내고, 서버는 상태 코드와 JSON 응답을 돌려줍니다.

| 구분 | 이번 수업의 값 | 뜻 |
|---|---|---|
| URL | `.../mealServiceDietInfo` | 어느 API 창구로 갈지 |
| `ATPT_OFCDC_SC_CODE` | `Q10` | 어느 교육청인지 |
| `SD_SCHUL_CODE` | `7140272` | 어느 학교인지 |
| `MLSV_FROM_YMD` | `20260624` | 조회 시작일 |
| `MLSV_TO_YMD` | `20260630` | 조회 종료일 |

### 2. 보내기 전에 완성된 주소를 확인한다

코드는 `requests.Request(...).prepare()`로 요청 미리보기를 만듭니다. 이 단계에서는 아직 서버에 요청하지 않습니다. 주소에 `SD_SCHUL_CODE=7140272`와 날짜가 제대로 들어갔는지 먼저 확인합니다.

### 3. 상태 코드를 보고 응답을 읽는다

`requests.get(..., timeout=15)`가 실제 요청을 보냅니다. 상태 코드 200은 서버가 요청을 정상적으로 처리했다는 뜻입니다. `raise_for_status()`는 404나 500 같은 HTTP 오류를 찾아내고, `response.json()`은 받은 JSON 글자를 Python 자료로 바꿉니다. JSON의 기호와 중첩 경로가 낯설면 부록 A로 돌아가 다시 연습합니다.

### 4. 실패할 때 사용할 자료도 준비한다

교실 인터넷이나 NEIS 서버가 잠시 불안정할 수 있습니다. 그래서 같은 학교·같은 기간의 예비 자료를 저장소에 함께 둡니다. 요청이 실패했다고 아무 날짜의 자료를 대신 쓰지 않고, 요청 기간과 겹칠 때만 전환합니다.

공식 안내에 따르면 인증키를 쓰지 않은 호출은 샘플 자료 5건으로 제한됩니다. 수업에서는 먼저 짧은 샘플 요청을 보내고, 별도 인증키를 사용할 때는 코드에 적지 않고 `NEIS_API_KEY` 환경 변수에서 읽습니다.


## 활동 전 생각


식당에 주문서를 낸다고 생각하고 빈칸을 채워 보세요.

| 주문서 항목 | 적을 내용 |
|---|---|
| 주문 창구 | 급식식단정보 API 주소 |
| 학교 | 학교명이 아니라 공식 학교 코드 `________` |
| 시작일 | `________` |
| 종료일 | `________` |
| 받고 싶은 형식 | `json` |

URL은 주문 창구이고 `params`는 주문서의 조건입니다. 같은 ‘남악고’라는 글자가 있더라도 공식 코드를 사용해야 정확한 학교를 구별할 수 있습니다.


## 예상하기

- 요청 미리보기 주소에는 학교 코드 7140272가 들어간다.
- 요청을 보내기 전 기본값에서는 `live_request_sent`가 `False`이다.
- 실시간 조회 실패 실험에서는 같은 기간의 남악고 예비 자료 5행을 사용한다.


## 활동 1. NEIS 요청 주소를 만들고 GET 함수 준비하기


### 코드 살펴보기


1. `NEIS_MEAL_URL`은 급식식단정보 API의 고정 주소입니다.<br>
2. `request_params`는 JSON 형식, 페이지, 학교, 기간을 키와 값으로 묶습니다.<br>
3. `preview_params`는 화면에 보여 줄 조건에서 인증키를 제외합니다.<br>
4. `requests.Request(...).prepare()`는 요청을 보내지 않고 안전한 미리보기 URL만 만듭니다.<br>
5. `requests.get(..., timeout=15)`가 실제 GET 요청을 보냅니다.<br>
6. `raise_for_status()`는 404나 500 같은 HTTP 오류를 성공으로 착각하지 않게 합니다.<br>
7. `response.json()`은 응답 JSON을 Python 사전과 목록으로 바꿉니다.<br>
8. 함수 안의 `safe_params`도 인증키를 뺀 응답 주소만 화면에 돌려줍니다.<br>
9. 인증키가 있더라도 URL이나 화면에 출력하지 않습니다.


In [1]:
import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
for candidate in (current_folder, *current_folder.parents):
    if (candidate / "jupyter_course" / "notebook_support.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. 프로젝트 최상위 폴더에서 "
        r".\.venv\Scripts\python.exe -m notebook 명령으로 다시 시작하세요."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jupyter_course.notebook_support import course_setup

setup = course_setup(PROJECT_ROOT)
PROJECT_ROOT = setup["root"]
raw_rows = setup["rows"]
meal_df = setup["frame"]
data_source = setup["source"]
print("프로젝트 폴더:", PROJECT_ROOT)
print("데이터 출처:", data_source)
print("급식 행 수:", len(raw_rows))

import os
import requests

NEIS_MEAL_URL = "https://open.neis.go.kr/hub/mealServiceDietInfo"
request_params = {
    "Type": "json",
    "pIndex": 1,
    "pSize": 5,
    "ATPT_OFCDC_SC_CODE": "Q10",
    "SD_SCHUL_CODE": "7140272",
    "MLSV_FROM_YMD": "20260624",
    "MLSV_TO_YMD": "20260630",
}

api_key = os.getenv("NEIS_API_KEY", "").strip()
if api_key:
    request_params["KEY"] = api_key

preview_params = {
    key: value for key, value in request_params.items() if key != "KEY"
}
prepared_request = requests.Request(
    "GET", NEIS_MEAL_URL, params=preview_params
).prepare()
prepared_request_url = prepared_request.url

def request_neis_meals(params, *, http_get=requests.get):
    response = http_get(NEIS_MEAL_URL, params=params, timeout=15)
    response.raise_for_status()
    safe_params = {key: value for key, value in params.items() if key != "KEY"}
    safe_url = requests.Request(
        "GET", NEIS_MEAL_URL, params=safe_params
    ).prepare().url
    return response.json(), safe_url, response.status_code

first_keys = sorted(raw_rows[0].keys())

print("요청 방식: GET")
print("요청 주소:", NEIS_MEAL_URL)
print("요청 조건 수:", len(request_params))
print("실제로 전송될 주소:", prepared_request_url)
print("인증키:", "환경 변수에서 읽음" if api_key else "샘플 호출(최대 5건)")


프로젝트 폴더: <프로젝트 폴더>
데이터 출처: 남악고 NEIS 예비 데이터
급식 행 수: 5
요청 방식: GET
요청 주소: https://open.neis.go.kr/hub/mealServiceDietInfo
요청 조건 수: 7
실제로 전송될 주소: https://open.neis.go.kr/hub/mealServiceDietInfo?Type=json&pIndex=1&pSize=5&ATPT_OFCDC_SC_CODE=Q10&SD_SCHUL_CODE=7140272&MLSV_FROM_YMD=20260624&MLSV_TO_YMD=20260630
인증키: 샘플 호출(최대 5건)


### 결과 해석하기

아직 서버에 보내지는 않았지만, 주소와 조건이 합쳐진 모습을 확인했습니다. 실제 요청 함수는 15초 안에 응답이 없으면 멈추고, HTTP 오류가 있으면 그대로 알려 줍니다.


## 활동 2. 실시간 우선·예비 자료 전환 연습


### 코드 살펴보기


1. `classroom_offline_demo`는 교실에서 연결 실패 상황을 재현합니다.<br>
2. `load_classroom_frame`은 실시간 조회가 실패하면 같은 기간의 예비 자료를 찾습니다.<br>
3. `try`와 `except NeisApiError`는 날짜가 맞지 않는 경우의 안내를 확인합니다.


In [2]:
from jupyter_course.notebook_support import load_classroom_frame
from neis_meal_ai.neis import NeisApiError

def classroom_offline_demo(_school, _start, _end):
    raise NeisApiError("교실용 연결 실패 실험")

classroom_frame, classroom_source = load_classroom_frame(
    PROJECT_ROOT,
    fetcher=classroom_offline_demo,
)
print("선택된 데이터 출처:", classroom_source)
print("분석 가능한 급식 행 수:", len(classroom_frame))

try:
    load_classroom_frame(
        PROJECT_ROOT,
        start="20260101",
        end="20260102",
        fetcher=classroom_offline_demo,
    )
except NeisApiError as error:
    print("날짜가 겹치지 않을 때의 안내:", error)

chapter_result = {
    "chapter": "01",
    "source": classroom_source,
    "raw_rows": len(classroom_frame),
    "first_keys": first_keys,
    "prepared_request_url": prepared_request_url,
    "live_request_sent": False,
}


선택된 데이터 출처: 남악고 예비 데이터 사용 · 실시간 조회 사유: 교실용 연결 실패 실험
분석 가능한 급식 행 수: 5
날짜가 겹치지 않을 때의 안내: 실시간 조회에 실패했고 요청 기간(20260101~20260102)과 겹치는 예비 데이터가 없습니다. 예비 데이터 기간: 20260624~20260630. 수업용 시연은 이 기간으로 조회하세요.


### 결과 해석하기

실시간 조회가 실패해도 같은 학교·같은 기간의 예비 자료가 있을 때만 전환됩니다. 기간이 겹치지 않으면 조용히 다른 날짜를 쓰지 않고 정확한 안내를 보여 줍니다.


## 탐구 활동

아래의 SEND_LIVE_REQUEST를 True로 바꾸면 방금 만든 함수가 공식 NEIS 서버에 GET 요청을 한 번 보냅니다. 먼저 False 상태에서 안내를 읽고, 요청 주소에 학교 코드와 날짜가 맞는지 확인한 뒤 True로 바꾸세요. 인증키가 없다면 공식 포털의 샘플 호출 제한에 따라 최대 5건만 받을 수 있습니다.

먼저 기본값으로 한 번 실행하세요. 그다음 표시된 값 하나만 바꾸고, 달라진 결과를 아래에 적습니다.


In [3]:
SEND_LIVE_REQUEST = False
live_request_sent = False

if SEND_LIVE_REQUEST:
    try:
        live_payload, response_url, status_code = request_neis_meals(request_params)
        live_request_sent = True
        print("응답 상태 코드:", status_code)
        print("보낸 주소:", response_url)
        if "mealServiceDietInfo" in live_payload:
            live_rows = live_payload["mealServiceDietInfo"][1]["row"]
            print("받은 급식 행 수:", len(live_rows))
            print("첫 급식 날짜:", live_rows[0]["MLSV_YMD"])
            print("첫 메뉴:", live_rows[0]["DDISH_NM"][:100] + "...")
        else:
            print("급식 행 대신 안내 응답이 왔습니다:", live_payload)
    except (requests.RequestException, ValueError, KeyError, IndexError) as error:
        print("요청 또는 JSON 읽기 안내:", error)
else:
    print("기본값은 전송하지 않습니다. 주소와 조건을 확인한 뒤 True로 바꾸세요.")

chapter_result["live_request_sent"] = live_request_sent


기본값은 전송하지 않습니다. 주소와 조건을 확인한 뒤 True로 바꾸세요.


### 내가 본 변화

- 내가 바꾼 값:  
- 화면에서 달라진 것:  
- 내 설명:


## 확인 문제

1. NEIS 요청에서 URL과 params는 각각 어떤 역할을 하나요?
2. 학교명 대신 학교 코드를 요청 조건에 넣는 까닭은 무엇인가요?
3. 요청을 보내기 전에 미리보기 주소에서 무엇을 확인해야 하나요?
4. `status_code`, `raise_for_status()`, `response.json()`은 차례로 무엇을 확인하거나 바꾸나요?
5. 실시간 API가 잠시 멈춰도 수업을 이어 갈 수 있는 이유는 무엇인가요?


## 정답과 해설


1. URL은 어느 API 창구로 갈지, params는 어느 학교의 어느 기간 자료를 달라고 할지 정합니다.<br>
2. 비슷하거나 같은 학교 이름이 있어도 공식 코드는 학교를 정확히 구별하는 식별값이기 때문입니다.<br>
3. 학교 코드 `7140272`, 교육청 코드 `Q10`, 시작일과 종료일이 맞는지 확인합니다.<br>
4. 상태 코드는 서버 처리 결과를 나타내고, `raise_for_status()`는 HTTP 오류를 확인하며, `response.json()`은 JSON 응답을 Python 자료로 바꿉니다.<br>
5. 같은 구조의 공식 NEIS 예비 데이터 5행을 프로젝트에 포함했기 때문입니다.


## 핵심 정리

- GET 요청은 API 주소와 params 조건을 합쳐 서버에 자료를 요청한다.
- 학교명 대신 공식 학교 코드로 정확한 학교를 구별한다.
- 보내기 전에 미리보기 주소의 학교 코드와 날짜를 확인한다.
- 응답은 상태 코드를 확인한 뒤 JSON으로 읽는다.
- 실시간 조회에 실패하면 같은 학교·같은 기간의 예비 자료로 전환한다.
- 남악고의 공식 코드는 Q10 / 7140272다.

### 다음 장

02장에서는 원본 문자열을 분석 가능한 표로 바꾸고 그래프로 읽습니다.


In [4]:
import json
print("__CHAPTER_RESULT__=" + json.dumps(chapter_result, ensure_ascii=False))


__CHAPTER_RESULT__={"chapter": "01", "source": "남악고 예비 데이터 사용 · 실시간 조회 사유: 교실용 연결 실패 실험", "raw_rows": 5, "first_keys": ["ATPT_OFCDC_SC_CODE", "ATPT_OFCDC_SC_NM", "CAL_INFO", "DDISH_NM", "LOAD_DTM", "MLSV_FGR", "MLSV_FROM_YMD", "MLSV_TO_YMD", "MLSV_YMD", "MMEAL_SC_CODE", "MMEAL_SC_NM", "NTR_INFO", "ORPLC_INFO", "SCHUL_NM", "SD_SCHUL_CODE"], "prepared_request_url": "https://open.neis.go.kr/hub/mealServiceDietInfo?Type=json&pIndex=1&pSize=5&ATPT_OFCDC_SC_CODE=Q10&SD_SCHUL_CODE=7140272&MLSV_FROM_YMD=20260624&MLSV_TO_YMD=20260630", "live_request_sent": false}
